

---


# **ESTADÍSTICA PARA ECONOMISTAS**
Unidad 1 -  Conceptos Básicos y Descripción de Variables

Datos: Gran Encuesta Integrada de Hogares (2018), módulo caracteristicas personas - Cúcuta (A.M)

*Mag. Sadan De la Cruz Almanza*

Fuente: https://microdatos.dane.gov.co/index.php/catalog/547



---




En este curso, utilizamos el lenguaje de R y Google Colab como herramientas para poder entender en la practica el funcionamiento de las estadísticas. Al no tratarse propiamente de un curso de programación, no se pretende que el estudiante "programe", se trata de aprender como funcionan los códigos para poder hacer uso de los mismos, el mundo de la programación requiere de una mayor profundización, incluso de cursos particulares. Como estrategia para avanzar en el aprendizaje de R, se agregaron algunas actividades para ser desarrolladas dentro del curso utilizando la IA Gemini para consultar sobre dudas puntuales en relación a los códigos. Los códigos suministrados, son una base y no un todo, es decir, en R existen muchas formas de hacer algo. Se suministra en ese caso un prompt sugerido para realizar las consultas:

Explica de forma exacta, en un párrafo y sin alucinaciones, la siguiente duda: [INSERTA TU FUNCIÓN/DUDA AQUÍ].

## **1. Paquetes y carga de datos**

In [ ]:
# 1.1 Paquetes

install.packages(c("haven", "dplyr", "ggplot2", "scales", "ggthemes"))

library(haven)      # Leer archivos .dta de Stata
library(dplyr)      # Manipulación de datos
library(ggplot2)    # Visualizaciones
library(scales)     # Formato de ejes (porcentajes, comas)

In [ ]:
# 1.2 Carga de datos

# Nota: para cargar datos en Google Colab, debemos ir al lado izquierdo del cuadernos al icono en forma de carpeta, abrir,y luego
#       arrastrar la base de datos. No debe incluir la base de datos dentro de alguna carpeta.

datos_personas <- read_dta("caracteristicas_personas_cucuta.DTA")

glimpse(datos_personas) # vista rápida del conjunto de datos

head(datos_personas) # resumen de la forma en como se visualiza el conjunto de datos

# Desde este momento, y hasta que termine de ejecutar el cuaderno de Google Colab, la versión inicial de la base (objeto) tendra como nombre "datos_personas"


## **2. Clasificación de variables y escalas de medición**



Una buena practica en el caso de estadística es la recodificación de las variables, este proceso consiste en modificar las variables en relación a la forma en como estas se presentan.

In [ ]:
# 2.1 selección de las variables

# En la práctica la selección de las variables depende del objetivo planteado en el análisis estadístico


   P6020  — Sexo     | Cualitativa   | Nominal  (1=Hombre, 2=Mujer)

   P6210  — Nivel edu| Cualitativa   | Ordinal  (1=Ninguno…9=Postgrado)

   P6040  — Edad     | Cuantitativa  | Razón    (años cumplidos; 0 absoluto)

   P6170  — Ingresos | Cuantitativa  | Razón    (pesos; 0 absoluto)

In [ ]:
datos1 <- datos_personas %>%
  select(P6020, P6210, P6040, P6170) # Función para seleccionar variables (debe tener presente escribir el nombre de la variable de manera correcta)

In [ ]:
# 2.2 recodificar las variables

# Una buena practica en el caso de estadística es la recodificación de las variables, este proceso consiste en modificar las variables respecto a su forma de presentación y otros cambios.

datos1 <- datos1 %>%
  mutate(                                                                        # La función mutate() puede transformar o crear variables
    sexo = factor(P6020, levels = 1:2,                                           # La función levels() ordena el número de las categorías
                  labels = c("Hombre", "Mujer")),                                # La función labels() crea una etiqueta para las categorías
    nivel_edu = factor(P6210,
                       levels = c(1, 2, 3, 4, 5, 6, 7, 9),
                       labels = c("Ninguno", "Preescolar", "Primaria",
                                  "Secundaria", "Media", "Superior incompleta",
                                  "Superior completa", "Postgrado")),
    edad  = as.numeric(P6040),
    ingreso = as.numeric(P6170)
  )

*Act1. Preguntale a Gemini*

¿Qué es una variable tipo "factor" en R?, ¿Qué es la función as.numeric()?

**Agregue una celda de texto en la parte inferior con el resumen de las respuestas entregadas por Gemini.**

## **3. Distribución de frecuencias**



### *3.1 Variable cualitativa nominal (sexo)*

In [ ]:
tabla_sexo <- datos1 %>%
  filter(!is.na(sexo)) %>%           # La función filter() permite "filtrar" los datos con observaciones especificas
  group_by(sexo) %>%
  summarise(
    fi = n(),                        # Frecuencia absoluta, la función n() calcula el total de observaciones
    .groups = "drop"
  ) %>%
  mutate(
    hi    = fi / sum(fi),            # Frecuencia relativa
    hi_pc = hi * 100,                # Frecuencia porcentual (%)
    Fi    = cumsum(fi),              # Frecuencia absoluta acumulada
    Hi    = cumsum(hi)               # Frecuencia relativa acumulada
  )

print(tabla_sexo)

# Fila de totales
total_sexo <- tibble(
  sexo  = "TOTAL",
  fi    = sum(tabla_sexo$fi),
  hi    = sum(tabla_sexo$hi),
  hi_pc = sum(tabla_sexo$hi_pc),
  Fi    = NA_real_,
  Hi    = NA_real_
)

tabla_sexo_print <- bind_rows(tabla_sexo, total_sexo) %>%
  mutate(across(where(is.numeric), ~ round(., 4)))

print(tabla_sexo_print)

*Act2. Preguntale a Gemini*

¿Qué hace la función !is.na()?, ¿Para que sirve el ! en R?, ¿Qué hace la función group_by()?, ¿Para que sirve el %>%?

**Agregue una celda de texto en la parte inferior con el resumen de las respuestas entregadas por Gemini.**

In [ ]:
g1 <- ggplot(tabla_sexo, aes(x = sexo, y = hi_pc, fill = sexo)) +
  geom_col(color = "white", width = 0.6) +
  geom_text(aes(label = sprintf("%.1f%%", hi_pc)),
            vjust = -0.5, size = 4.5, fontface = "bold") +
  scale_fill_manual(values = c("Hombre" = "#4472C4", "Mujer" = "#ED7D31")) +
  scale_y_continuous(limits = c(0, 60), labels = function(x) paste0(x, "%")) +
  labs(
    title    = "Distribución porcentual por Sexo",
    subtitle = "GEIH 2018 — Cúcuta (AM) | Variable nominal",
    x        = "Sexo",
    y        = "Frecuencia relativa (%)",
    fill     = NULL,
    caption  = "Fuente: DANE — GEIH 2018"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none",
        plot.title      = element_text(face = "bold"))

print(g1)

In [ ]:
g2 <- ggplot(tabla_sexo, aes(x = "", y = hi_pc, fill = sexo)) +
  geom_col(color = "white", width = 1) +
  coord_polar("y") +
  geom_text(aes(label = sprintf("%s\n%.1f%%", sexo, hi_pc)),
            position = position_stack(vjust = 0.5),
            size = 5, fontface = "bold", color = "white") +
  scale_fill_manual(values = c("Hombre" = "#4472C4", "Mujer" = "#ED7D31")) +
  labs(
    title    = "Composición por Sexo — Gráfico de Torta",
    subtitle = "GEIH 2018 — Cúcuta (A.M.) | Variable nominal",
    fill     = NULL,
    caption  = "Fuente: DANE — GEIH 2018"
  ) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        plot.title      = element_text(face = "bold", hjust = 0.5),
        plot.subtitle   = element_text(hjust = 0.5))

print(g2)

### *3.2 Variable cualitativa ordinal (nivel educativo)*

In [ ]:
tabla_edu <- datos1 %>%
  filter(!is.na(nivel_edu)) %>%
  group_by(nivel_edu) %>%
  summarise(fi = n(), .groups = "drop") %>%
  arrange(nivel_edu) %>%           # Respeta el orden de la escala ordinal
  mutate(
    hi    = fi / sum(fi),
    hi_pc = hi * 100,
    Fi    = cumsum(fi),
    Hi    = cumsum(hi)
  )

total_edu <- tibble(
  nivel_edu = "TOTAL",
  fi    = sum(tabla_edu$fi),
  hi    = 1,
  hi_pc = 100,
  Fi    = NA_real_,
  Hi    = NA_real_
)

bind_rows(
  tabla_edu %>% mutate(nivel_edu = as.character(nivel_edu)),
  total_edu
) %>%
  mutate(across(where(is.numeric), ~ round(., 4))) %>%
  print(n = Inf)

In [ ]:
tabla_edu_plot <- tabla_edu %>%
  arrange(desc(fi)) %>%
  mutate(
    nivel_edu = factor(nivel_edu, levels = nivel_edu),  # Ordenar descendente
    Fi_pct    = cumsum(hi_pc)
  )

g3 <- ggplot(tabla_edu_plot, aes(x = nivel_edu, y = hi_pc)) +
  geom_col(fill = "#4472C4", color = "white", width = 0.7) +
  geom_line(aes(y = Fi_pct / 2, group = 1),   # Eje secundario simplificado
            color = "#ED7D31", linewidth = 1, linetype = "dashed") +
  geom_point(aes(y = Fi_pct / 2), color = "#ED7D31", size = 3) +
  geom_text(aes(label = sprintf("%.1f%%", hi_pc)),
            vjust = -0.5, size = 3.5, fontface = "bold") +
  scale_y_continuous(
    name     = "Frecuencia relativa (%)",
    limits   = c(0, 55),
    sec.axis = sec_axis(~ . * 2, name = "Frecuencia acumulada (%)",
                        labels = function(x) paste0(x, "%"))
  ) +
  labs(
    title    = "Distribución por Nivel Educativo",
    subtitle = "GEIH 2018 — Cúcuta (AM) | Variable ordinal",
    x        = "Nivel educativo",
    caption  = "Barras: frecuencia relativa (%) | Línea naranja: acumulado (%)\nFuente: DANE — GEIH 2018"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 35, hjust = 1),
    plot.title  = element_text(face = "bold")
  )

print(g3)

**Pregunta 1.**

En el caso de las variables cualitativas ordinales, si el orden importa, ¿por qué en la gráfica no se presenta con ese enfoque?

### *3.3 Variable cuantitativa por intervalos (edad)*

La **regla de Sturges** para la construcción de intervalos:

Es un criterio utilizado en estadística para determinar el número de clases o intervalos necesarios para construir una tabla de frecuencias a partir de un conjunto de datos.



In [ ]:
edad_vec <- datos1 %>%
  filter(!is.na(edad), edad >= 0, edad <= 120) %>%
  pull(edad)

n_edad <- length(edad_vec)
k      <- ceiling(1 + 3.322 * log10(n_edad))   # Número de clases (regla de Sturges)
rango  <- max(edad_vec) - min(edad_vec)
ancho  <- ceiling(rango / k)                    # Amplitud del intervalo

cat(sprintf("  n = %d  |  k (Sturges) = %d  |  Rango = %d  |  Amplitud = %d años\n\n",
            n_edad, k, rango, ancho))

breaks_edad <- seq(min(edad_vec), min(edad_vec) + k * ancho, by = ancho) # Construcción de los intervalos

tabla_edad <- tibble(edad = edad_vec) %>%
  mutate(
    clase = cut(edad, breaks = breaks_edad, right = FALSE, include.lowest = TRUE)
  ) %>%
  group_by(clase) %>%
  summarise(fi = n(), .groups = "drop") %>%
  mutate(
    Xi    = (as.numeric(sub("\\[(.+),.*", "\\1", clase)) +              # Marca de clase Xi = punto medio del intervalo
             as.numeric(sub(".*,(.+)[\\]\\)]", "\\1", clase))) / 2,
    hi    = fi / sum(fi),
    hi_pc = hi * 100,
    Fi    = cumsum(fi),
    Hi    = cumsum(hi)
  ) %>%
  select(clase, Xi, fi, hi, hi_pc, Fi, Hi)

total_edad <- tibble(
  clase = "TOTAL", Xi = NA_real_,
  fi = sum(tabla_edad$fi), hi = 1, hi_pc = 100,
  Fi = NA_real_, Hi = NA_real_
)

bind_rows(
  tabla_edad %>% mutate(clase = as.character(clase)),
  total_edad
) %>%
  mutate(across(where(is.numeric), ~ round(., 4))) %>%
  print(n = Inf)

cat("\n")

In [ ]:
breaks_manual <- c(0, 5, 12, 18, 28, 40, 60, 120)   # Creación de intervalos

etiquetas_manual <- c(
  "Primera infancia (0–4)",
  "Infancia (5–11)",
  "Adolescencia (12–17)",
  "Juventud (18–27)",
  "Adultez temprana (28–39)",
  "Adultez media (40–59)",
  "Adultez mayor (60+)"
) # Se sugiere este etiquetado, pero esto dependerá de cada caso en la practica.

tabla_edad_manual <- tibble(edad = edad_vec) %>%
  mutate(
    clase = cut(edad,
                breaks         = breaks_manual,
                labels         = etiquetas_manual,
                right          = FALSE,       # Intervalos cerrados a la izquierda [Li, Ls)
                include.lowest = TRUE)
  ) %>%
  group_by(clase) %>%
  summarise(fi = n(), .groups = "drop") %>%
  mutate(
    # Marca de clase Xi = punto medio del intervalo
    Li    = breaks_manual[-length(breaks_manual)],
    Ls    = breaks_manual[-1],
    Xi    = (Li + Ls) / 2,
    hi    = fi / sum(fi),                       # Frecuencia relativa
    hi_pc = hi * 100,                           # Frecuencia porcentual (%)
    Fi    = cumsum(fi),                         # Frec. absoluta acumulada
    Hi    = cumsum(hi)                          # Frec. relativa acumulada
  ) %>%
  select(clase, Li, Ls, Xi, fi, hi, hi_pc, Fi, Hi)

# Fila de totales
total_manual <- tibble(
  clase = "TOTAL", Li = NA_real_, Ls = NA_real_, Xi = NA_real_,
  fi    = sum(tabla_edad_manual$fi),
  hi    = 1, hi_pc = 100,
  Fi    = NA_real_, Hi = NA_real_
)

tabla_edad_manual

In [ ]:
g4 <- ggplot(data.frame(edad = edad_vec), aes(x = edad)) +
  geom_histogram(breaks  = breaks_edad,
                 fill    = "#4472C4",
                 color   = "white",
                 closed  = "left") +
  geom_vline(xintercept = mean(edad_vec),
             color = "#FF0000", linewidth = 1, linetype = "dashed") +
  geom_vline(xintercept = median(edad_vec),
             color = "#FF8C00", linewidth = 1, linetype = "dotted") +
  annotate("text", x = mean(edad_vec) + 2, y = Inf, vjust = 2,
           label = sprintf("Media = %.1f", mean(edad_vec)),
           color = "#FF0000", size = 3.8) +
  annotate("text", x = median(edad_vec) - 2, y = Inf, vjust = 4,
           label = sprintf("Mediana = %.0f", median(edad_vec)),
           color = "#FF8C00", size = 3.8, hjust = 1) +
  scale_y_continuous(labels = comma) +
  labs(
    title    = "Distribución de la Edad — Histograma de Frecuencias",
    subtitle = sprintf("GEIH 2018 — Cúcuta (AM) | Regla de Sturges: k = %d clases, amplitud = %d años",
                       k, ancho),
    x        = "Edad (años)",
    y        = "Frecuencia absoluta (fi)",
    caption  = "Fuente: DANE — GEIH 2018"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

print(g4)

In [ ]:
g5 <- ggplot(data.frame(edad = edad_vec), aes(x = edad)) +
  geom_histogram(aes(y = after_stat(count / sum(count) * 100)),
                 breaks = breaks_edad,
                 fill   = "#70AD47",
                 color  = "white",
                 closed = "left") +
  scale_y_continuous(labels = function(x) paste0(x, "%")) +
  labs(
    title    = "Distribución de la Edad — Frecuencia Relativa (%)",
    subtitle = "GEIH 2018 — Cúcuta (AM) | Variable cuantitativa discreta",
    x        = "Edad (años)",
    y        = "Frecuencia relativa (hi × 100)",
    caption  = "Fuente: DANE — GEIH 2018"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

print(g5)

Este contenido hace parte de las unidades del curso estadística para ciencias económicas del departamento de economía de la Universidad de Pamplona. Sugerencias y comentarios al correo sadan.de@unipamplona.edu.co